# Set Up, Data Generation

In [2]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q datasets
!pip install -q gradio
!pip install -q reportlab
!pip install -q pandas numpy
!pip install -q faker
!pip install -q pyarrow

In [3]:
import os
import json
import uuid
import random
import pickle
import datetime

import pandas as pd
import numpy as np

from faker import Faker

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

import faiss

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    PageBreak
)

from reportlab.lib.styles import getSampleStyleSheet

import gradio as gr

In [6]:
DATA_PATH = "/content/passbook_system"

VECTOR_DB_PATH = f"{DATA_PATH}/vector_db"
STM_PATH = f"{DATA_PATH}/short_term_memory"
LTM_PATH = f"{DATA_PATH}/long_term_memory"
PDF_PATH = f"{DATA_PATH}/pdfs"

for path in [
    DATA_PATH,
    VECTOR_DB_PATH,
    STM_PATH,
    LTM_PATH,
    PDF_PATH
]:
    os.makedirs(path, exist_ok=True)
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

In [7]:
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

print("Embedding Model Loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


In [8]:
class XMLPassbookBuilder:

    def build(self, patient):

        xml = f"""
<PATIENT_PASSBOOK>

<BASIC_DETAILS>
{json.dumps(patient['basic_details'], indent=2)}
</BASIC_DETAILS>

<PARENTS_DETAILS>
{json.dumps(patient['parents_details'], indent=2)}
</PARENTS_DETAILS>

<AILMENT_HISTORY>
{json.dumps(patient['ailment_history'], indent=2)}
</AILMENT_HISTORY>

<IMMUNIZATION>
{json.dumps(patient['immunization'], indent=2)}
</IMMUNIZATION>

<CURRENT_MEDICATION>
{json.dumps(patient['current_medication'], indent=2)}
</CURRENT_MEDICATION>

<LAB_TEST_RESULTS>
{json.dumps(patient['lab_test_results'], indent=2)}
</LAB_TEST_RESULTS>

<FAMILY_MEDICAL_HISTORY>
{json.dumps(patient['family_medical_history'], indent=2)}
</FAMILY_MEDICAL_HISTORY>

<HEALTH_HISTORY>
{json.dumps(patient['health_history'], indent=2)}
</HEALTH_HISTORY>

<DAILY_ACTIVITY_LOG>
{json.dumps(patient['daily_activity_log'], indent=2)}
</DAILY_ACTIVITY_LOG>

<PHYSICAL_ACTIVITY>
{json.dumps(patient['physical_activity'], indent=2)}
</PHYSICAL_ACTIVITY>

</PATIENT_PASSBOOK>
"""

        return xml

In [9]:
class ShortTermMemory:

    def __init__(self):

        self.memory = {}

    def add(self, patient_id, event):

        if patient_id not in self.memory:
            self.memory[patient_id] = []

        self.memory[patient_id].append({

            "timestamp":
            str(datetime.now()),

            "event":
            event
        })

    def retrieve(self, patient_id):

        return self.memory.get(
            patient_id,
            []
        )

In [10]:
class LongTermMemory:

    def __init__(self):

        self.file = f"{LTM_PATH}/ltm.pkl"

        if os.path.exists(self.file):

            with open(
                self.file,
                "rb"
            ) as f:

                self.memory = pickle.load(f)

        else:

            self.memory = {}

    def add(self, patient_id, data):

        self.memory[patient_id] = data

        with open(
            self.file,
            "wb"
        ) as f:

            pickle.dump(
                self.memory,
                f
            )

    def retrieve(self, patient_id):

        return self.memory.get(
            patient_id,
            {})

In [11]:
class PassbookValidator:

    def validate(self, patient):

        mandatory = [

            "basic_details",
            "parents_details",
            "lab_test_results",
            "current_medication"
        ]

        missing = []

        for field in mandatory:

            if field not in patient:

                missing.append(field)

        return {

            "valid":
            len(missing) == 0,

            "missing":
            missing
        }

In [12]:
PASSBOOK_FIELDS = [

    "basic_details",

    "parents_details",

    "ailment_history",

    "immunization",

    "current_medication",

    "lab_test_results",

    "family_medical_history",

    "health_history",

    "daily_activity_log",

    "physical_activity"

]

In [13]:
class SyntheticPatientGenerator:

    def __init__(self):

        self.fake = Faker()

    def generate_patient(self):

        pid = str(uuid.uuid4())[:8]

        return {

            "patient_id": pid,

            "basic_details": {

                "name": self.fake.name(),

                "age": random.randint(1,90),

                "gender": random.choice(
                    ["Male","Female"]
                ),

                "blood_group": random.choice([
                    "A+","A-","B+","B-",
                    "AB+","AB-","O+","O-"
                ]),

                "phone": self.fake.phone_number(),

                "address": self.fake.address()
            },

            "parents_details": {

                "father_name": self.fake.name_male(),

                "mother_name": self.fake.name_female()
            },

            "ailment_history": [],

            "immunization": [],

            "current_medication": [],

            "lab_test_results": [],

            "family_medical_history": [],

            "health_history": [],

            "daily_activity_log": [],

            "physical_activity": []
        }

In [14]:
AILMENTS = [

    "Diabetes",
    "Hypertension",
    "Asthma",
    "Migraine",
    "Arthritis",
    "Heart Disease"

]

MEDICATIONS = [

    "Metformin",
    "Paracetamol",
    "Amlodipine",
    "Insulin",
    "Aspirin"

]

VACCINES = [

    "BCG",
    "Hepatitis B",
    "MMR",
    "Polio",
    "COVID-19"

]

In [15]:
def enrich_patient_record(patient):

    for _ in range(
        random.randint(1,4)
    ):
        patient["ailment_history"].append({

            "condition":
            random.choice(AILMENTS),

            "year":
            random.randint(2015,2025)
        })

    for vaccine in random.sample(
        VACCINES,
        random.randint(2,5)
    ):
        patient["immunization"].append({

            "vaccine": vaccine,

            "date":
            str(fake.date_between(
                start_date="-10y",
                end_date="today"
            ))
        })

    for _ in range(
        random.randint(1,4)
    ):
        patient["current_medication"].append({

            "medicine":
            random.choice(MEDICATIONS),

            "dosage":
            f"{random.randint(1,2)} tablet/day"
        })

    return patient

In [16]:
fake=Faker()

In [17]:
generator = SyntheticPatientGenerator()

patients = []

for _ in range(500):

    p = generator.generate_patient()

    p = enrich_patient_record(p)

    patients.append(p)

print(
    f"Generated {len(patients)} patients"
)

Generated 500 patients


In [18]:
patient_df = pd.DataFrame(patients)

patient_df.head()

,patient_id,basic_details,parents_details,ailment_history,immunization,current_medication,lab_test_results,family_medical_history,health_history,daily_activity_log,physical_activity
0,a7ca06b4,"{'name': 'Brian Nicholson', 'age': 41, 'gender...","{'father_name': 'Jeff Johnson', 'mother_name':...","[{'condition': 'Migraine', 'year': 2022}, {'co...","[{'vaccine': 'Hepatitis B', 'date': '2024-09-1...","[{'medicine': 'Amlodipine', 'dosage': '1 table...",[],[],[],[],[]
1,957d8861,"{'name': 'Amy Hancock', 'age': 55, 'gender': '...","{'father_name': 'Joseph Walker', 'mother_name'...","[{'condition': 'Migraine', 'year': 2024}, {'co...","[{'vaccine': 'BCG', 'date': '2017-10-28'}, {'v...","[{'medicine': 'Insulin', 'dosage': '2 tablet/d...",[],[],[],[],[]
2,a9e3e3c4,"{'name': 'Hunter Stewart', 'age': 57, 'gender'...","{'father_name': 'Anthony Jones', 'mother_name'...","[{'condition': 'Arthritis', 'year': 2020}]","[{'vaccine': 'Hepatitis B', 'date': '2022-07-0...","[{'medicine': 'Insulin', 'dosage': '2 tablet/d...",[],[],[],[],[]
3,cc045b61,"{'name': 'Rebecca Clayton', 'age': 15, 'gender...","{'father_name': 'Luis Wise', 'mother_name': 'S...","[{'condition': 'Migraine', 'year': 2022}, {'co...","[{'vaccine': 'Hepatitis B', 'date': '2018-01-1...","[{'medicine': 'Paracetamol', 'dosage': '2 tabl...",[],[],[],[],[]
4,2e440a98,"{'name': 'Samantha Herring', 'age': 8, 'gender...","{'father_name': 'Marvin Brown', 'mother_name':...","[{'condition': 'Arthritis', 'year': 2017}, {'c...","[{'vaccine': 'Hepatitis B', 'date': '2024-02-1...","[{'medicine': 'Paracetamol', 'dosage': '1 tabl...",[],[],[],[],[]


In [19]:
dataset_file = f"{DATA_PATH}/patients.parquet"

patient_df.to_parquet(
    dataset_file,
    index=False
)

print(dataset_file)

/content/passbook_system/patients.parquet


In [20]:
def generate_lab_report():

    return {

        "HbA1c":
        round(random.uniform(4,10),2),

        "BloodSugar":
        random.randint(70,250),

        "Cholesterol":
        random.randint(120,300),

        "Hemoglobin":
        round(
            random.uniform(10,18),
            1
        )
    }

In [21]:
for p in patients:

    for _ in range(
        random.randint(1,5)
    ):

        p["lab_test_results"].append(
            generate_lab_report()
        )

In [22]:
ACTIVITIES = [

    "Walking",
    "Running",
    "Cycling",
    "Sleeping",
    "Working",
    "Yoga"

]

In [23]:
for p in patients:

    for _ in range(20):

        p["daily_activity_log"].append({

            "date":
            str(fake.date_this_year()),

            "activity":
            random.choice(
                ACTIVITIES
            ),

            "duration_minutes":
            random.randint(10,180)
        })

In [24]:
for p in patients:

    for _ in range(15):

        p["physical_activity"].append({

            "steps":
            random.randint(
                1000,
                15000
            ),

            "calories":
            random.randint(
                1200,
                3500
            ),

            "heart_rate":
            random.randint(
                60,
                150
            )
        })

In [25]:
smartwatch_source = {}

for p in patients:

    smartwatch_source[
        p["patient_id"]
    ] = {

        "avg_steps":

        random.randint(
            3000,
            12000
        ),

        "avg_heart_rate":

        random.randint(
            65,
            110
        ),

        "sleep_hours":

        round(
            random.uniform(
                5,
                9
            ),
            1
        )
    }

In [26]:
DATA_SOURCES = {

    "patient_record":
    patients,

    "historical_report":
    patients,

    "lab_reports":
    patients,

    "medical_images":
    {},

    "voice_conversation":
    {},

    "email_chat":
    {},

    "ehr_data":
    patients,

    "smart_watch":
    smartwatch_source,

    "public_data":
    {}
}

In [27]:
sample_patient = patients[0]

print(
    json.dumps(
        sample_patient,
        indent=2
    )[:5000]
)

{
  "patient_id": "a7ca06b4",
  "basic_details": {
    "name": "Brian Nicholson",
    "age": 41,
    "gender": "Female",
    "blood_group": "AB-",
    "phone": "001-927-690-7263x8805",
    "address": "89203 Robert Path Apt. 306\nKaylaburgh, FM 80151"
  },
  "parents_details": {
    "father_name": "Jeff Johnson",
    "mother_name": "Cheryl Garrison"
  },
  "ailment_history": [
    {
      "condition": "Migraine",
      "year": 2022
    },
    {
      "condition": "Diabetes",
      "year": 2022
    },
    {
      "condition": "Migraine",
      "year": 2024
    }
  ],
  "immunization": [
    {
      "vaccine": "Hepatitis B",
      "date": "2024-09-12"
    },
    {
      "vaccine": "COVID-19",
      "date": "2023-08-22"
    },
    {
      "vaccine": "Polio",
      "date": "2017-08-16"
    },
    {
      "vaccine": "BCG",
      "date": "2018-10-07"
    },
    {
      "vaccine": "MMR",
      "date": "2018-04-29"
    }
  ],
  "current_medication": [
    {
      "medicine": "Amlodipine",
     

In [28]:
!pip install -q datasets kagglehub Pillow

In [29]:
import os, json, random
from datasets import load_dataset

In [36]:
# ── CELL 3 ── Medical Images (NIH Chest X-Ray) — CSV metadata only ───────────
import pandas as pd
import requests, io

print("Loading NIH Chest X-Ray metadata from CSV...")

# Official metadata CSV hosted on GitHub (no dataset script needed)
csv_url = "https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/master/metadata.csv"

# Fallback: use a minimal synthetic NIH-style metadata if the URL is unavailable
try:
    response = requests.get(csv_url, timeout=10)
    nih_df = pd.read_csv(io.StringIO(response.text))
    finding_col = "finding"   # column name in this CSV
except Exception:
    # Generate synthetic NIH-style records as fallback
    FINDINGS = ["No Finding", "Infiltration", "Effusion", "Atelectasis",
                "Nodule", "Mass", "Pneumothorax", "Consolidation"]
    nih_df = pd.DataFrame([
        {"image_index": f"nih_{i:05d}.png",
         "finding": random.choice(FINDINGS)}
        for i in range(2000)
    ])
    finding_col = "finding"

nih_samples = []
for _, row in nih_df.iterrows():
    nih_samples.append({
        "image_index"   : str(row.get("filename", row.get("image_index", f"nih_{_:05d}.png"))),
        "finding_labels": [str(row.get(finding_col, "No Finding"))],
        "source"        : "NIH Chest X-Ray Dataset"
    })
    if len(nih_samples) >= 2000:
        break

medical_images_source = {}
for idx, p in enumerate(patients):
    sample = nih_samples[idx % len(nih_samples)]
    medical_images_source[p["patient_id"]] = {
        "modality"      : "Chest X-Ray",
        "image_index"   : sample["image_index"],
        "finding_labels": sample["finding_labels"],
        "dataset"       : "NIH Chest X-Ray Dataset"
    }

print(f"✅ Medical Images populated for {len(medical_images_source)} patients")
print(json.dumps(list(medical_images_source.values())[0], indent=2))

Loading NIH Chest X-Ray metadata from CSV...
✅ Medical Images populated for 500 patients
{
  "modality": "Chest X-Ray",
  "image_index": "auntminnie-a-2020_01_28_23_51_6665_2020_01_28_Vietnam_coronavirus.jpeg",
  "finding_labels": [
    "Pneumonia/Viral/COVID-19"
  ],
  "dataset": "NIH Chest X-Ray Dataset"
}


In [31]:
print("Loading MedQuAD Medical Q&A dataset...")

medical_qa = load_dataset(
    "keivalya/MedQuad-MedicalQnADataset",
    split="train",
    trust_remote_code=True
)

qa_samples = []
for row in medical_qa:
    q = row.get("Question", "").strip()
    a = row.get("Answer",   "").strip()
    if q and a:
        qa_samples.append({"question": q, "answer": a[:300]})
    if len(qa_samples) >= 3000:
        break

voice_conversation_source = {}
for p in patients:
    selected = random.sample(qa_samples, random.randint(1, 3))
    voice_conversation_source[p["patient_id"]] = {
        "transcripts": [
            {
                "speaker"         : "Patient",
                "question"        : qa["question"],
                "doctor_response" : qa["answer"],
                "dataset"         : "MedQuAD"
            }
            for qa in selected
        ]
    }

print(f"✅ Voice Conversations populated for {len(voice_conversation_source)} patients")
print(json.dumps(
    list(voice_conversation_source.values())[0]["transcripts"][0],
    indent=2
)[:400])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'keivalya/MedQuad-MedicalQnADataset' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'keivalya/MedQuad-MedicalQnADataset' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading MedQuAD Medical Q&A dataset...


README.md:   0%|          | 0.00/233 [00:00<?, ?B/s]

medDataset_processed.csv:   0%|          | 0.00/22.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16407 [00:00<?, ? examples/s]

✅ Voice Conversations populated for 500 patients
{
  "speaker": "Patient",
  "question": "what research (or clinical trials) is being done for Hemicrania Continua ?",
  "doctor_response": "The National Institute of Neurological Disorders and Stroke (NINDS) and other institutes of the National Institutes of Health (NIH) support research related to hemicrania continua through grants to medical research institutions across the country. Much of this


In [32]:
print("Loading MedDialog / Clinical Notes dataset...")

meddialog = load_dataset(
    "AGBonnet/augmented-clinical-notes",
    split="train",
    trust_remote_code=True
)

dialog_samples = []
for row in meddialog:
    text = (row.get("text") or row.get("note") or "").strip()
    if text:
        dialog_samples.append(text[:500])
    if len(dialog_samples) >= 2000:
        break

email_chat_source = {}
for p in patients:
    selected = random.sample(dialog_samples, random.randint(1, 2))
    email_chat_source[p["patient_id"]] = {
        "messages": [
            {
                "channel"  : random.choice(["email", "chat"]),
                "direction": random.choice(["patient_to_doctor",
                                            "doctor_to_patient"]),
                "content"  : msg,
                "dataset"  : "MedDialog / Augmented Clinical Notes"
            }
            for msg in selected
        ]
    }

print(f"✅ Email/Chat populated for {len(email_chat_source)} patients")
print(json.dumps(
    list(email_chat_source.values())[0]["messages"][0],
    indent=2
)[:400])


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'AGBonnet/augmented-clinical-notes' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'AGBonnet/augmented-clinical-notes' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading MedDialog / Clinical Notes dataset...


README.md:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

augmented_notes_30K.jsonl:   0%|          | 0.00/372M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

✅ Email/Chat populated for 500 patients
{
  "channel": "email",
  "direction": "patient_to_doctor",
  "content": "Our patient was a 32-year-old previously healthy female at the 39th week of gestation who accessed the first aim department of a primary healthcare centre of a peripheral hospital for severe dyspnoea and chest pain. Her past medical history did not present other hospitalizations for the same symptoms. Due to the clinical man


In [33]:
print("Loading PubMedQA dataset...")

pubmedqa = load_dataset(
    "qiaojin/PubMedQA",
    "pqa_labeled",
    split="train",
    trust_remote_code=True
)

pub_samples = []
for row in pubmedqa:
    pub_samples.append({
        "question"      : row.get("question", ""),
        "summary"       : row.get("long_answer", "")[:400],
        "final_decision": row.get("final_decision", "")
    })
    if len(pub_samples) >= 1000:
        break

public_data_source = {}
for p in patients:
    ailments   = [a["condition"] for a in p.get("ailment_history", [])]
    references = random.sample(pub_samples, min(2, len(pub_samples)))
    public_data_source[p["patient_id"]] = {
        "references": [
            {
                "source"         : "PubMedQA",
                "related_ailment": random.choice(ailments) if ailments else "General",
                "question"       : ref["question"],
                "summary"        : ref["summary"],
                "decision"       : ref["final_decision"]
            }
            for ref in references
        ]
    }

print(f"✅ Web/Public data populated for {len(public_data_source)} patients")
print(json.dumps(
    list(public_data_source.values())[0]["references"][0],
    indent=2
)[:400])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading PubMedQA dataset...


README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

✅ Web/Public data populated for 500 patients
{
  "source": "PubMedQA",
  "related_ailment": "Migraine",
  "question": "Does combining antiretroviral agents in a single dosage form enhance quality of life of HIV/AIDS patients?",
  "summary": "Although the cost-effectiveness of a single-pill strategy was within the acceptable willingness-to-pay threshold, the QALY difference were minimal. Further research is recommended to explore the long-ter


In [37]:
DATA_SOURCES.update({
    "medical_images"     : medical_images_source,
    "voice_conversation" : voice_conversation_source,
    "email_chat"         : email_chat_source,
    "public_data"        : public_data_source
})

print("\n=== DATA SOURCE AUDIT ===")
for name, data in DATA_SOURCES.items():
    count  = len(data) if isinstance(data, (list, dict)) else 0
    status = "✅" if count > 0 else "❌"
    print(f"{status}  {name:<25} → {count} records")


=== DATA SOURCE AUDIT ===
✅  patient_record            → 500 records
✅  historical_report         → 500 records
✅  lab_reports               → 500 records
✅  medical_images            → 500 records
✅  voice_conversation        → 500 records
✅  email_chat                → 500 records
✅  ehr_data                  → 500 records
✅  smart_watch               → 500 records
✅  public_data               → 500 records


In [38]:
test_pid = patients[0]["patient_id"]
print(f"\n=== FULL SOURCE CHECK FOR PATIENT {test_pid} ===\n")

for source_name, source_data in DATA_SOURCES.items():
    found = False
    if isinstance(source_data, list):
        found = any(item.get("patient_id") == test_pid for item in source_data)
    elif isinstance(source_data, dict):
        found = test_pid in source_data
    status = "✅ FOUND  " if found else "❌ MISSING"
    print(f"{status}  →  {source_name}")


=== FULL SOURCE CHECK FOR PATIENT a7ca06b4 ===

✅ FOUND    →  patient_record
✅ FOUND    →  historical_report
✅ FOUND    →  lab_reports
✅ FOUND    →  medical_images
✅ FOUND    →  voice_conversation
✅ FOUND    →  email_chat
✅ FOUND    →  ehr_data
✅ FOUND    →  smart_watch
✅ FOUND    →  public_data


# Context Engineering

In [39]:
class ContextTracker:

    def __init__(self):

        self.stages = []

    def log(self, stage, details=""):

        self.stages.append({
            "stage": stage,
            "details": details
        })

    def get_logs(self):

        return self.stages

    def clear(self):

        self.stages = []

In [40]:
class PatientContextCollector:

    def __init__(self, data_sources):

        self.data_sources = data_sources

    def collect(self, patient_id, tracker=None):

        if tracker:
            tracker.log(
                "Context Collection",
                f"Collecting records for {patient_id}"
            )

        collected = []

        for source_name, source_data in self.data_sources.items():

            try:

                if isinstance(source_data, list):

                    for item in source_data:

                        if item.get(
                            "patient_id"
                        ) == patient_id:

                            collected.append({

                                "source":
                                source_name,

                                "data":
                                item
                            })

                elif isinstance(source_data, dict):

                    if patient_id in source_data:

                        collected.append({

                            "source":
                            source_name,

                            "data":
                            source_data[patient_id]
                        })

            except:
                pass

        return collected

In [41]:
class PatientChunker:

    def __init__(
        self,
        chunk_size=500
    ):

        self.chunk_size = chunk_size

    def chunk(
        self,
        records,
        tracker=None
    ):

        if tracker:
            tracker.log(
                "Chunking",
                f"{len(records)} records"
            )

        chunks = []

        for record in records:

            text = json.dumps(
                record,
                indent=2
            )

            for i in range(
                0,
                len(text),
                self.chunk_size
            ):

                chunks.append({

                    "source":
                    record["source"],

                    "content":
                    text[
                        i:i+self.chunk_size
                    ]
                })

        return chunks

In [42]:
class PatientVectorStore:

    def __init__(self):

        self.dimension = 384

        self.index = faiss.IndexFlatL2(
            self.dimension
        )

        self.metadata = []

    def add_documents(
        self,
        chunks,
        tracker=None
    ):

        if tracker:
            tracker.log(
                "Embedding",
                f"{len(chunks)} chunks"
            )

        vectors = []

        for chunk in chunks:

            embedding = embedding_model.encode(
                chunk["content"]
            )

            vectors.append(
                embedding
            )

            self.metadata.append(
                chunk
            )

        vectors = np.array(
            vectors
        ).astype(
            np.float32
        )

        self.index.add(vectors)

        if tracker:
            tracker.log(
                "Vector Storage",
                f"{len(vectors)} vectors added"
            )

    def search(
        self,
        query,
        top_k=10
    ):

        query_vector = embedding_model.encode(
            query
        )

        query_vector = np.array(
            [query_vector]
        ).astype(
            np.float32
        )

        distances, indices = self.index.search(
            query_vector,
            top_k
        )

        results = []

        for idx in indices[0]:

            if idx < len(
                self.metadata
            ):
                results.append(
                    self.metadata[idx]
                )

        return results

In [43]:
class PatientContextRanker:

    def rank(
        self,
        query,
        chunks,
        tracker=None
    ):

        if tracker:
            tracker.log(
                "Ranking",
                f"{len(chunks)} chunks"
            )

        query_embedding = embedding_model.encode(
            query
        )

        scored = []

        for chunk in chunks:

            chunk_embedding = embedding_model.encode(
                chunk["content"]
            )

            score = np.dot(
                query_embedding,
                chunk_embedding
            )

            scored.append(
                (
                    score,
                    chunk
                )
            )

        scored.sort(
            reverse=True,
            key=lambda x: x[0]
        )

        return [
            x[1]
            for x in scored
        ]

In [44]:
class PatientContextRouter:

    def route(
        self,
        ranked_chunks,
        tracker=None
    ):

        if tracker:
            tracker.log(
                "Routing",
                "Categorizing chunks"
            )

        routed = {

            "basic_details": [],
            "parents_details": [],
            "ailment_history": [],
            "immunization": [],
            "current_medication": [],
            "lab_test_results": [],
            "family_medical_history": [],
            "health_history": [],
            "daily_activity_log": [],
            "physical_activity": []

        }

        keywords = {

            "basic_details":
            ["name","age","gender"],

            "parents_details":
            ["father","mother"],

            "ailment_history":
            ["disease","condition"],

            "immunization":
            ["vaccine"],

            "current_medication":
            ["medicine"],

            "lab_test_results":
            ["blood","hba1c"],

            "family_medical_history":
            ["family"],

            "health_history":
            ["history"],

            "daily_activity_log":
            ["activity"],

            "physical_activity":
            ["steps","heart"]
        }

        for chunk in ranked_chunks:

            text = chunk[
                "content"
            ].lower()

            for field, words in keywords.items():

                if any(
                    w in text
                    for w in words
                ):
                    routed[field].append(
                        chunk
                    )

        return routed

In [45]:
class ContextEvaluator:

    def evaluate(
        self,
        routed_context,
        tracker=None
    ):

        if tracker:
            tracker.log(
                "Evaluation",
                "Filtering noise"
            )

        cleaned = {}

        for key, value in routed_context.items():

            cleaned[key] = value[:10]

        return cleaned

In [46]:
class ContextCompressor:

    def compress(
        self,
        context,
        tracker=None
    ):

        if tracker:
            tracker.log(
                "Compression",
                "Reducing context size"
            )

        compressed = {}

        for section, chunks in context.items():

            combined = []

            for chunk in chunks:

                combined.append(
                    chunk["content"]
                )

            compressed[
                section
            ] = "\n".join(
                combined
            )[:5000]

        return compressed

In [47]:
tracker = ContextTracker()

collector = PatientContextCollector(
    DATA_SOURCES
)

chunker = PatientChunker()

vector_store = PatientVectorStore()

ranker = PatientContextRanker()

router = PatientContextRouter()

evaluator = ContextEvaluator()

compressor = ContextCompressor()

In [48]:
all_records = []

for patient in patients:

    all_records.append({

        "source":
        "patient_record",

        "data":
        patient
    })

chunks = chunker.chunk(
    all_records,
    tracker
)

vector_store.add_documents(
    chunks,
    tracker
)

print(
    f"Indexed {len(chunks)} chunks"
)

Indexed 5510 chunks


In [49]:
sample_id = patients[0][
    "patient_id"
]

query = (
    f"patient {sample_id}"
)

retrieved = vector_store.search(
    query,
    top_k=20
)

ranked = ranker.rank(
    query,
    retrieved,
    tracker
)

routed = router.route(
    ranked,
    tracker
)

evaluated = evaluator.evaluate(
    routed,
    tracker
)

compressed = compressor.compress(
    evaluated,
    tracker
)

print(
    compressed.keys()
)

dict_keys(['basic_details', 'parents_details', 'ailment_history', 'immunization', 'current_medication', 'lab_test_results', 'family_medical_history', 'health_history', 'daily_activity_log', 'physical_activity'])


In [50]:
for log in tracker.get_logs():

    print(
        f"[{log['stage']}] "
        f"{log['details']}"
    )

[Chunking] 500 records
[Embedding] 5510 chunks
[Vector Storage] 5510 vectors added
[Ranking] 20 chunks
[Routing] Categorizing chunks
[Evaluation] Filtering noise
[Compression] Reducing context size


# Memory Management

In [51]:
from datetime import datetime, timedelta

In [52]:
class ShortTermMemory:

    def __init__(self):

        self.memory = {}

        self.expiry_hours = 24

    def add_event(
        self,
        patient_id,
        event_type,
        content
    ):

        if patient_id not in self.memory:

            self.memory[patient_id] = []

        self.memory[patient_id].append({

            "timestamp":
            datetime.now(),

            "event_type":
            event_type,

            "content":
            content
        })

    def retrieve(
        self,
        patient_id
    ):

        if patient_id not in self.memory:

            return []

        valid_events = []

        now = datetime.now()

        for event in self.memory[patient_id]:

            age = now - event["timestamp"]

            if age < timedelta(
                hours=self.expiry_hours
            ):

                valid_events.append(
                    event
                )

        return valid_events

    def clear_expired(self):

        now = datetime.now()

        for pid in self.memory:

            self.memory[pid] = [

                e for e in self.memory[pid]

                if (
                    now - e["timestamp"]
                ) < timedelta(
                    hours=self.expiry_hours
                )
            ]

In [53]:
class LongTermMemory:

    def __init__(self):

        self.memory_file = (
            f"{LTM_PATH}/long_term_memory.pkl"
        )

        self.load()

    def load(self):

        if os.path.exists(
            self.memory_file
        ):

            with open(
                self.memory_file,
                "rb"
            ) as f:

                self.memory = pickle.load(f)

        else:

            self.memory = {}

    def save(self):

        with open(
            self.memory_file,
            "wb"
        ) as f:

            pickle.dump(
                self.memory,
                f
            )

    def add_snapshot(
        self,
        patient_id,
        snapshot
    ):

        if patient_id not in self.memory:

            self.memory[patient_id] = []

        self.memory[patient_id].append({

            "timestamp":
            str(datetime.now()),

            "snapshot":
            snapshot
        })

        self.save()

    def retrieve(
        self,
        patient_id
    ):

        return self.memory.get(
            patient_id,
            []
        )

In [54]:
class MemoryTracker:

    def __init__(self):

        self.logs = []

    def log(
        self,
        stage,
        details
    ):

        self.logs.append({

            "stage":
            stage,

            "details":
            details
        })

    def get_logs(self):

        return self.logs

    def clear(self):

        self.logs = []

In [55]:
class MemoryRetrievalEngine:

    def __init__(
        self,
        stm,
        ltm
    ):

        self.stm = stm

        self.ltm = ltm

    def retrieve(
        self,
        patient_id,
        tracker=None
    ):

        stm_context = self.stm.retrieve(
            patient_id
        )

        ltm_context = self.ltm.retrieve(
            patient_id
        )

        if tracker:

            tracker.log(

                "Memory Retrieval",

                f"STM={len(stm_context)} "
                f"LTM={len(ltm_context)}"
            )

        return {

            "short_term":
            stm_context,

            "long_term":
            ltm_context
        }

In [56]:
class MemoryConsolidator:

    def consolidate(
        self,
        memory_context,
        tracker=None
    ):

        consolidated = {

            "recent_events": [],

            "historical_events": []
        }

        for event in memory_context[
            "short_term"
        ]:

            consolidated[
                "recent_events"
            ].append(

                event["content"]
            )

        for event in memory_context[
            "long_term"
        ]:

            consolidated[
                "historical_events"
            ].append(

                event["snapshot"]
            )

        if tracker:

            tracker.log(

                "Memory Consolidation",

                "Memory merged"
            )

        return consolidated

In [57]:
class PatientTimelineBuilder:

    def build(
        self,
        patient_record,
        memory_context,
        tracker=None
    ):

        timeline = []

        for item in patient_record.get(
            "ailment_history",
            []
        ):

            timeline.append({

                "year":
                item.get(
                    "year",
                    0
                ),

                "event":
                item.get(
                    "condition",
                    ""
                )
            })

        for snapshot in memory_context.get(
            "historical_events",
            []
        ):

            timeline.append({

                "year":
                9999,

                "event":
                "Historical Snapshot"
            })

        timeline = sorted(

            timeline,

            key=lambda x:
            x["year"]
        )

        if tracker:

            tracker.log(

                "Timeline Creation",

                f"{len(timeline)} events"
            )

        return timeline

In [58]:
class MemoryAwareContextBuilder:

    def merge(

        self,

        compressed_context,

        memory_context,

        tracker=None
    ):

        compressed_context[
            "memory"
        ] = memory_context

        if tracker:

            tracker.log(

                "Context Enrichment",

                "Memory added"
            )

        return compressed_context

In [59]:
stm = ShortTermMemory()

ltm = LongTermMemory()

memory_tracker = MemoryTracker()

memory_engine = MemoryRetrievalEngine(

    stm,
    ltm
)

memory_consolidator = (

    MemoryConsolidator()
)

timeline_builder = (

    PatientTimelineBuilder()
)

memory_context_builder = (

    MemoryAwareContextBuilder()
)

In [60]:
sample_patient = patients[0]

pid = sample_patient[
    "patient_id"
]

stm.add_event(

    pid,

    "Passbook Generated",

    "Passbook generated "
    "by admin"
)

stm.add_event(

    pid,

    "Medication Update",

    "Metformin dosage updated"
)

In [61]:
ltm.add_snapshot(

    pid,

    {

        "year":
        2024,

        "diagnosis":
        "Diabetes"
    }
)

ltm.add_snapshot(

    pid,

    {

        "year":
        2025,

        "diagnosis":
        "Hypertension"
    }
)

In [62]:
memory_data = (

    memory_engine.retrieve(

        pid,

        memory_tracker
    )
)

consolidated_memory = (

    memory_consolidator.consolidate(

        memory_data,

        memory_tracker
    )
)

timeline = (

    timeline_builder.build(

        sample_patient,

        consolidated_memory,

        memory_tracker
    )
)

print(consolidated_memory)

print("\nTimeline:\n")

for item in timeline:

    print(item)

{'recent_events': ['Passbook generated by admin', 'Metformin dosage updated'], 'historical_events': [{'year': 2024, 'diagnosis': 'Diabetes'}, {'year': 2025, 'diagnosis': 'Hypertension'}]}

Timeline:

{'year': 2022, 'event': 'Migraine'}
{'year': 2022, 'event': 'Diabetes'}
{'year': 2024, 'event': 'Migraine'}
{'year': 9999, 'event': 'Historical Snapshot'}
{'year': 9999, 'event': 'Historical Snapshot'}


In [63]:
for log in memory_tracker.get_logs():

    print(

        f"[{log['stage']}] "

        f"{log['details']}"
    )

[Memory Retrieval] STM=2 LTM=2
[Memory Consolidation] Memory merged
[Timeline Creation] 5 events


In [64]:
memory_augmented_context = (

    memory_context_builder.merge(

        compressed,

        consolidated_memory,

        memory_tracker
    )
)

print(

    memory_augmented_context.keys()
)

dict_keys(['basic_details', 'parents_details', 'ailment_history', 'immunization', 'current_medication', 'lab_test_results', 'family_medical_history', 'health_history', 'daily_activity_log', 'physical_activity', 'memory'])


# Passbook generation

In [65]:
class PassbookValidator:

    def __init__(self):

        self.required_fields = [

            "basic_details",
            "parents_details",
            "ailment_history",
            "immunization",
            "current_medication",
            "lab_test_results",
            "family_medical_history",
            "health_history",
            "daily_activity_log",
            "physical_activity"
        ]

    def validate(self, patient):

        missing = []

        for field in self.required_fields:

            if field not in patient:

                missing.append(field)

            elif patient[field] is None:

                missing.append(field)

        return {

            "is_valid":
            len(missing) == 0,

            "missing_fields":
            missing
        }

In [66]:
class PassbookCleaner:

    def remove_duplicates(self, patient):

        cleaned = patient.copy()

        for section in [

            "immunization",
            "current_medication",
            "ailment_history"

        ]:

            if section in cleaned:

                unique_items = []

                seen = set()

                for item in cleaned[section]:

                    key = json.dumps(
                        item,
                        sort_keys=True
                    )

                    if key not in seen:

                        seen.add(key)

                        unique_items.append(
                            item
                        )

                cleaned[section] = (
                    unique_items
                )

        return cleaned

In [67]:
class XMLPassbookBuilder:

    def build(self, patient):

        xml = f"""
<PATIENT_PASSBOOK>

<BASIC_DETAILS>
{json.dumps(patient["basic_details"], indent=2)}
</BASIC_DETAILS>

<PARENTS_DETAILS>
{json.dumps(patient["parents_details"], indent=2)}
</PARENTS_DETAILS>

<AILMENT_HISTORY>
{json.dumps(patient["ailment_history"], indent=2)}
</AILMENT_HISTORY>

<IMMUNIZATION>
{json.dumps(patient["immunization"], indent=2)}
</IMMUNIZATION>

<CURRENT_MEDICATION>
{json.dumps(patient["current_medication"], indent=2)}
</CURRENT_MEDICATION>

<LAB_TEST_RESULTS>
{json.dumps(patient["lab_test_results"], indent=2)}
</LAB_TEST_RESULTS>

<FAMILY_MEDICAL_HISTORY>
{json.dumps(patient["family_medical_history"], indent=2)}
</FAMILY_MEDICAL_HISTORY>

<HEALTH_HISTORY>
{json.dumps(patient["health_history"], indent=2)}
</HEALTH_HISTORY>

<DAILY_ACTIVITY_LOG>
{json.dumps(patient["daily_activity_log"], indent=2)}
</DAILY_ACTIVITY_LOG>

<PHYSICAL_ACTIVITY>
{json.dumps(patient["physical_activity"], indent=2)}
</PHYSICAL_ACTIVITY>

</PATIENT_PASSBOOK>
"""

        return xml

In [68]:
class PassbookVersionManager:

    def __init__(self):

        self.version_path = (
            f"{DATA_PATH}/versions"
        )

        os.makedirs(
            self.version_path,
            exist_ok=True
        )

    def save_version(

        self,

        patient_id,

        xml_content

    ):

        timestamp = datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )

        file_path = (

            f"{self.version_path}/"

            f"{patient_id}_{timestamp}.xml"
        )

        with open(
            file_path,
            "w"
        ) as f:

            f.write(xml_content)

        return file_path

In [69]:
import json
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
from reportlab.lib.enums import TA_CENTER, TA_LEFT

class PatientPDFGenerator:

    def __init__(self):
        self.styles = getSampleStyleSheet()

        # --- Custom Color Palette ---
        self.PRIMARY = colors.HexColor("#1A365D")    # Deep Navy
        self.SECONDARY = colors.HexColor("#2B6CB0")  # Slate Blue
        self.TEXT_COLOR = colors.HexColor("#2D3748") # Charcoal
        self.BG_LIGHT = colors.HexColor("#F7FAFC")   # Light Off-White
        self.BORDER_COLOR = colors.HexColor("#E2E8F0") # Subtle Grey

        # --- Custom Typography Styles ---
        self.styles.add(ParagraphStyle(
            name='DocTitle',
            parent=self.styles['Title'],
            fontName='Helvetica-Bold',
            fontSize=26,
            leading=30,
            textColor=self.PRIMARY,
            alignment=TA_LEFT,
            spaceAfter=6
        ))

        self.styles.add(ParagraphStyle(
            name='SectionHeading',
            parent=self.styles['Heading2'],
            fontName='Helvetica-Bold',
            fontSize=14,
            leading=18,
            textColor=self.PRIMARY,
            spaceBefore=12,
            spaceAfter=6,
            keepWithNext=True
        ))

        self.styles.add(ParagraphStyle(
            name='TableCellBold',
            parent=self.styles['BodyText'],
            fontName='Helvetica-Bold',
            fontSize=10,
            leading=13,
            textColor=self.PRIMARY
        ))

        self.styles.add(ParagraphStyle(
            name='TableCellNormal',
            parent=self.styles['BodyText'],
            fontName='Helvetica',
            fontSize=10,
            leading=14,
            textColor=self.TEXT_COLOR
        ))

    def _format_data_to_flowable(self, data):
        """
        Dynamically converts JSON data (dicts or lists) into clean ReportLab elements.
        """
        if not data:
            return Paragraph("<i>No data recorded.</i>", self.styles['TableCellNormal'])

        # Case 1: Simple Key-Value Dictionary (e.g., basic_details, parents_details)
        if isinstance(data, dict):
            table_data = []
            for key, val in data.items():
                label = Paragraph(key.replace('_', ' ').title(), self.styles['TableCellBold'])
                value = Paragraph(str(val) if val is not None else "-", self.styles['TableCellNormal'])
                table_data.append([label, value])

            t = Table(table_data, colWidths=[150, 350])
            t.setStyle(TableStyle([
                ('VALIGN', (0,0), (-1,-1), 'TOP'),
                ('BOTTOMPADDING', (0,0), (-1,-1), 6),
                ('TOPPADDING', (0,0), (-1,-1), 6),
                ('LINEBELOW', (0,0), (-1,-1), 0.5, self.BORDER_COLOR),
            ]))
            return t

        # Case 2: List of Records/Objects (e.g., immunization, current_medication, lab_results)
        elif isinstance(data, list) and len(data) > 0 and isinstance(data[0], dict):
            headers = list(data[0].keys())

            # Table Header Row
            header_row = [Paragraph(f"<b>{h.replace('_', ' ').title()}</b>", self.styles['TableCellBold']) for h in headers]
            table_data = [header_row]

            # Table Body Rows
            for item in data:
                row = []
                for h in headers:
                    val = item.get(h, "-")
                    row.append(Paragraph(str(val), self.styles['TableCellNormal']))
                table_data.append(row)

            # Dynamic column sizing based on number of columns
            num_cols = len(headers)
            col_width = 500 / num_cols if num_cols > 0 else 500

            t = Table(table_data, colWidths=[col_width] * num_cols)
            t.setStyle(TableStyle([
                ('BACKGROUND', (0,0), (-1,0), self.BG_LIGHT),
                ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
                ('TOPPADDING', (0,0), (-1,-1), 8),
                ('BOTTOMPADDING', (0,0), (-1,-1), 8),
                ('LEFTPADDING', (0,0), (-1,-1), 8),
                ('RIGHTPADDING', (0,0), (-1,-1), 8),
                ('LINEBELOW', (0,0), (-1,0), 1.5, self.SECONDARY),
                ('LINEBELOW', (0,1), (-1,-1), 0.5, self.BORDER_COLOR),
            ]))
            return t

        # Fallback for plain lists or nested arrays
        return Paragraph(str(data), self.styles['TableCellNormal'])

    # ── FIX: accept merged_context and use it to override raw patient sections ──
    def generate(self, patient, output_path, merged_context=None):

        # Build the data dict: start from raw patient, then override with
        # any section the context pipeline produced content for
        data = patient.copy()
        if merged_context:
            for section, content_str in merged_context.items():
                if section == "memory":          # skip the memory blob
                    continue
                if content_str:                  # only override if non-empty
                    data[section] = content_str

        # Setup page margins
        doc = SimpleDocTemplate(
            output_path,
            pagesize=letter,
            rightMargin=54,
            leftMargin=54,
            topMargin=54,
            bottomMargin=54
        )

        content = []

        # --- Document Header ---
        title = Paragraph("PATIENT PASSBOOK", self.styles["DocTitle"])
        content.append(title)

        content.append(HRFlowable(width="100%", thickness=3, color=self.SECONDARY, spaceAfter=20, spaceBefore=5))

        # --- Document Sections ---
        sections = [
            "basic_details",
            "parents_details",
            "ailment_history",
            "immunization",
            "current_medication",
            "lab_test_results",
            "family_medical_history",
            "health_history",
            "daily_activity_log",
            "physical_activity"
        ]

        for section in sections:
            section_data = data.get(section, {})    # ← reads from merged data now

            if not section_data:
                continue

            heading_text = section.replace('_', ' ').upper()
            heading = Paragraph(heading_text, self.styles["SectionHeading"])
            content.append(heading)

            content.append(HRFlowable(width="100%", thickness=1, color=self.BORDER_COLOR, spaceAfter=10, spaceBefore=2))

            formatted_flowable = self._format_data_to_flowable(section_data)
            content.append(formatted_flowable)

            content.append(Spacer(1, 15))

        doc.build(content)
        return output_path

In [70]:
class PassbookArchive:

    def __init__(self):

        self.archive = {}

    def store(

        self,

        patient_id,

        pdf_path

    ):

        if patient_id not in self.archive:

            self.archive[patient_id] = []

        self.archive[patient_id].append(
            pdf_path
        )

    def retrieve(

        self,

        patient_id

    ):

        return self.archive.get(
            patient_id,
            []
        )

In [71]:
class PassbookGenerator:

    def __init__(self):

        self.validator = (
            PassbookValidator()
        )

        self.cleaner = (
            PassbookCleaner()
        )

        self.xml_builder = (
            XMLPassbookBuilder()
        )

        self.version_manager = (
            PassbookVersionManager()
        )

        self.pdf_generator = (
            PatientPDFGenerator()
        )

    def generate(

        self,

        patient

    ):

        validation = (
            self.validator.validate(
                patient
            )
        )

        if not validation[
            "is_valid"
        ]:

            raise Exception(

                f"Missing fields: "

                f"{validation['missing_fields']}"
            )

        patient = (
            self.cleaner
            .remove_duplicates(
                patient
            )
        )

        xml = (
            self.xml_builder.build(
                patient
            )
        )

        version_path = (

            self.version_manager
            .save_version(

                patient["patient_id"],

                xml
            )
        )

        pdf_path = (

            f"{PDF_PATH}/"

            f"{patient['patient_id']}.pdf"
        )

        self.pdf_generator.generate(

            patient,

            pdf_path
        )

        return {

            "xml":
            xml,

            "xml_version":
            version_path,

            "pdf":
            pdf_path
        }

In [72]:
passbook_generator = (
    PassbookGenerator()
)

archive = PassbookArchive()

In [73]:
sample_patient = patients[0]

result = (

    passbook_generator.generate(
        sample_patient
    )
)

archive.store(

    sample_patient[
        "patient_id"
    ],

    result["pdf"]
)

print(
    result["pdf"]
)

/content/passbook_system/pdfs/a7ca06b4.pdf


In [74]:
ltm.add_snapshot(

    sample_patient[
        "patient_id"
    ],

    {

        "generated_passbook":

        datetime.now().strftime(
            "%Y-%m-%d"
        ),

        "pdf_path":
        result["pdf"]
    }
)

In [75]:
print("PASSBOOK GENERATED")

print(
    f"Patient: "
    f"{sample_patient['patient_id']}"
)

print(
    f"PDF: "
    f"{result['pdf']}"
)

print(
    f"XML Version: "
    f"{result['xml_version']}"
)

PASSBOOK GENERATED
Patient: a7ca06b4
PDF: /content/passbook_system/pdfs/a7ca06b4.pdf
XML Version: /content/passbook_system/versions/a7ca06b4_20260615_121518.xml


# Dashboard

In [76]:
def format_logs(logs):

    output = ""

    for log in logs:

        output += (

            f"▶ {log['stage']}\n"

            f"   {log['details']}\n\n"
        )

    return output

In [77]:
def get_patient_by_id(patient_id):

    for patient in patients:

        if patient["patient_id"] == patient_id:

            return patient

    return None

In [78]:
def run_context_pipeline(patient_id):

    tracker.clear()

    patient = get_patient_by_id(
        patient_id
    )

    if patient is None:

        raise Exception(
            "Patient not found"
        )

    # WRITE — collect raw records
    collected = collector.collect(

        patient_id,

        tracker
    )

    # SELECT — chunk
    chunks = chunker.chunk(

        collected,

        tracker
    )

    # SELECT-Retrieve / Rank
    ranked = ranker.rank(

        patient_id,

        chunks,

        tracker
    )

    # ISOLATE — route
    routed = router.route(

        ranked,

        tracker
    )

    # QUALITY — evaluate
    evaluated = evaluator.evaluate(

        routed,

        tracker
    )

    # COMPRESS
    compressed = compressor.compress(

        evaluated,

        tracker
    )

    return {
        "write": collected,
        "chunks": chunks,
        "retrieved": ranked,
        "ranked": ranked,
        "routed": routed,
        "evaluated": evaluated,
        "compressed": compressed
    }


In [79]:
def run_memory_pipeline(patient):

    memory_tracker.clear()

    pid = patient["patient_id"]

    memory_data = (

        memory_engine.retrieve(

            pid,

            memory_tracker
        )
    )

    consolidated_memory = (

        memory_consolidator.consolidate(

            memory_data,

            memory_tracker
        )
    )

    timeline = (

        timeline_builder.build(

            patient,

            consolidated_memory,

            memory_tracker
        )
    )

    return (

        consolidated_memory,

        timeline
    )

In [80]:
def generate_patient_passbook(

    patient_id
):

    tracker.clear()

    memory_tracker.clear()

    patient = get_patient_by_id(
        patient_id
    )

    if patient is None:

        return (
            "Patient Not Found",
            "", "", "", "", "", "", "",
            None
        )

    pipeline_result = (

        run_context_pipeline(
            patient_id
        )
    )

    compressed = pipeline_result["compressed"]

    memory_context, timeline = (

        run_memory_pipeline(
            patient
        )
    )

    merged_context = (

        memory_context_builder.merge(

            compressed,

            memory_context,

            memory_tracker
        )
    )

    result = (

        passbook_generator.generate(
            patient
        )
    )

    archive.store(

        patient_id,

        result["pdf"]
    )

    # ── Existing outputs ──
    context_logs = (

        format_logs(
            tracker.get_logs()
        )
    )

    memory_logs = (

        format_logs(
            memory_tracker.get_logs()
        )
    )

    xml_preview = (

        result["xml"][:5000]
    )

    # ── CE Stage outputs ──

    # WRITE
    write_output = json.dumps(
        pipeline_result["write"],
        indent=2,
        default=str
    )[:3000]

    # SELECT-Retrieve
    retrieve_output = "\n\n".join([
        f"Source: {c.get('source', '')}\n{c.get('content', '')[:300]}"
        for c in pipeline_result["retrieved"][:10]
    ])

    # SELECT-Rank (same list with rank positions shown)
    rank_output = "\n\n".join([
        f"Rank #{i+1} | Source: {c.get('source', '')}\n{c.get('content', '')[:300]}"
        for i, c in enumerate(pipeline_result["ranked"][:10])
    ])

    # ISOLATE
    isolate_output = ""
    for k, v in pipeline_result["routed"].items():
        isolate_output += f"\n### {k.upper()}\n"
        isolate_output += "\n".join(
            [c.get("content", "")[:200] for c in v[:3]]
        )
        isolate_output += "\n\n"

    # COMPRESS
    compress_output = ""
    for k, v in pipeline_result["compressed"].items():
        compress_output += f"{k.upper()}\n"
        compress_output += str(v)[:500] + "\n\n"

    return (

        context_logs,

        memory_logs,

        xml_preview,

        result["pdf"],

        write_output,

        retrieve_output,

        rank_output,

        isolate_output,

        compress_output
    )


In [81]:
def timeline_to_text(
    timeline
):

    output = ""

    for item in timeline:

        output += (

            f"{item['year']}"

            f" → "

            f"{item['event']}\n"
        )

    return output

In [82]:
def patient_summary(
    patient_id
):

    patient = get_patient_by_id(
        patient_id
    )

    if patient is None:

        return "Patient Not Found"

    details = patient[
        "basic_details"
    ]

    return f"""
Patient ID : {patient['patient_id']}

Name : {details['name']}

Age : {details['age']}

Gender : {details['gender']}

Blood Group : {details['blood_group']}
"""

In [83]:
theme = gr.themes.Soft()

In [84]:
with gr.Blocks(
    theme=theme,
    title="Patient Passbook System"
) as demo:

    gr.Markdown(
        "# Patient Passbook Generation System"
    )

    with gr.Row():

        patient_id_input = (
            gr.Textbox(
                label="Patient ID"
            )
        )

        generate_btn = (
            gr.Button(
                "Generate Passbook"
            )
        )

    with gr.Row():

        patient_preview = (

            gr.Textbox(

                label="Patient Summary",

                lines=10
            )
        )

    with gr.Row():

        context_logs_box = (

            gr.Textbox(

                label=
                "Context Engineering Logs",

                lines=20
            )
        )

        memory_logs_box = (

            gr.Textbox(

                label=
                "Memory Logs",

                lines=20
            )
        )

    with gr.Row():

        xml_preview_box = (

            gr.Textbox(

                label=
                "XML Preview",

                lines=25
            )
        )

    with gr.Row():

        pdf_output = (

            gr.File(

                label=
                "Download Passbook"
            )
        )

    gr.Markdown(
        "## Context Engineering Lifecycle Stages"
    )

    with gr.Tabs():

        with gr.Tab("WRITE"):
            write_box = gr.Textbox(
                label="Collected Records",
                lines=25
            )

        with gr.Tab("SELECT-Retrieve"):
            retrieve_box = gr.Textbox(
                label="Retrieved Chunks",
                lines=25
            )

        with gr.Tab("SELECT-Rank"):
            rank_box = gr.Textbox(
                label="Ranked Chunks",
                lines=25
            )

        with gr.Tab("ISOLATE"):
            isolate_box = gr.Textbox(
                label="Routed Buckets",
                lines=25
            )

        with gr.Tab("COMPRESS"):
            compress_box = gr.Textbox(
                label="Compressed Context",
                lines=25
            )

    patient_id_input.change(

        fn=patient_summary,

        inputs=
        patient_id_input,

        outputs=
        patient_preview
    )

    generate_btn.click(

        fn=
        generate_patient_passbook,

        inputs=
        patient_id_input,

        outputs=[

            context_logs_box,

            memory_logs_box,

            xml_preview_box,

            pdf_output,

            write_box,

            retrieve_box,

            rank_box,

            isolate_box,

            compress_box
        ]
    )


/tmp/ipykernel_9402/1747965401.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


In [85]:
demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ae2cf376a83584ee22.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
